In [0]:
%run ./00_common

In [0]:
import pandas as pd

def collect_latest_history_pdf(table_name: str, run_id: str) -> pd.DataFrame:
    hist_pdf = spark.sql(f"DESCRIBE HISTORY {table_name}").limit(1).toPandas()

    if hist_pdf.empty:
        return pd.DataFrame(columns=[
            "audit_ts", "run_id", "table_name", "version",
            "operation", "operation_ts", "num_output_rows", "user_name"
        ])

    row = hist_pdf.iloc[0]
    operation_metrics = row.get("operationMetrics", None)
    num_output_rows = None

    if isinstance(operation_metrics, dict):
        num_output_rows = operation_metrics.get("numOutputRows")

    return pd.DataFrame([{
        "audit_ts": utc_now_naive(),
        "run_id": run_id,
        "table_name": table_name,
        "version": row.get("version"),
        "operation": row.get("operation"),
        "operation_ts": row.get("timestamp"),
        "num_output_rows": str(num_output_rows) if num_output_rows is not None else None,
        "user_name": row.get("userName") if "userName" in hist_pdf.columns else None
    }])